# Explore "tips.csv"

Generated for a selected file in Catalog Explorer. This notebook shows how to do the following: - viewing file metadata - parsing documents with ai_parse_document - analyzing images with ai_query

In [0]:
# System-filled
FILE_PATH = "/Volumes/workspace/default/firstvolume/tips.csv"

## 1. Show file metadata

In [0]:
df = spark.read.csv(FILE_PATH, header=True, inferSchema=True)
df.show()

## Handling Categorical Features

In [0]:
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol="sex", outputCol="sex_indexed")
df_r = indexer.fit(df).transform(df)

In [0]:
df_r.show()

In [0]:
indexer = StringIndexer(inputCols=["smoker", "day", "time"], outputCols=["smoker_induced", "day_indexed", "time_indexed"])
df_s = indexer.fit(df_r).transform(df_r)
df_s.show()

In [0]:
df_s.columns

In [0]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=["tip","size","sex_indexed","smoker_induced","day_indexed","time_indexed"], outputCol="Independent feature")
output = assembler.transform(df_s)

In [0]:
output.select("Independent feature").show()

In [0]:
finalized_data = output.select("Independent feature", "total_bill")
finalized_data.show()

In [0]:
from pyspark.ml.regression import LinearRegression
# train data, test data split
train_data, test_data = finalized_data.randomSplit([0.75,0.25])
regressor = LinearRegression(featuresCol="Independent feature", labelCol="total_bill")
regressor = regressor.fit(train_data)

In [0]:
regressor.coefficients

In [0]:
regressor.intercept

In [0]:
pred_results = regressor.evaluate(test_data)

In [0]:
pred_results.predictions.show()

In [0]:
pred_results.r2, pred_results.meanAbsoluteError, pred_results.meanSquaredError